In [ ]:
import json
import numpy as np
import ollama
import chromadb
from pathlib import Path

cleaned_documents_file = Path("../data/processed/cleaned_documents.json")
chunked_documents_file = Path("../data/processed/chunked_documents.json")
embedding_file = Path("../data/processed/embeddings.npy")
chroma_path = Path("../data/chroma")

print("Setup complete")

In [ ]:
#loading documents

with open(cleaned_documents_file, "r", encoding="utf-8") as f:
    cleaned_documents = json.load(f)

print("Total documents:", len(cleaned_documents))

In [ ]:
#loading chunks 

with open(chunked_documents_file, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print("Total chunks:", len(all_chunks))

In [ ]:

#loading embeddings 

embeddings_array = np.load(embedding_file)

print("Embeddings shape:", embeddings_array.shape)

In [ ]:
#checking if chunks and embeddings are loaded and matches
print("Chunks:", len(all_chunks))
print("Embeddings:", len(embeddings_array))

assert len(all_chunks) == len(embeddings_array)

print("Chunk and embedding counts match")

In [ ]:
#creating a chromadb client to store the embeddings

client = chromadb.PersistentClient(
    path=str(chroma_path)
)

print("ChromaDB client created successfully")

In [ ]:
#creating a collection to store into chromadb

collection = client.get_or_create_collection(
    name="documents"
)

print("Collection:", collection.name)
print("Current count:", collection.count())

In [ ]:
#storing the chunks in batched of size = 100

batch_size = 100

for start in range(0, len(all_chunks), batch_size):

    end = min(start + batch_size, len(all_chunks))

    batch_chunks = all_chunks[start:end]
    batch_embeddings = embeddings_array[start:end]

    collection.add(
        ids=[chunk["chunk_id"] for chunk in batch_chunks],
        embeddings=batch_embeddings.tolist(),
        documents=[chunk["text"] for chunk in batch_chunks],
        metadatas=[
            {
                "doc_id": chunk["doc_id"],
                "source": chunk["source"],
                "chunk_index": chunk["chunk_index"]
            }
            for chunk in batch_chunks
        ]
    )

    print(f"Inserted {end}/{len(all_chunks)} chunks")

In [ ]:
#checking the collection count

print("Chroma collection count:", collection.count())

In [ ]:
#Fetching the first chunk for verifying 

result = collection.get(
    ids=[all_chunks[0]["chunk_id"]],
    include=["embeddings", "documents", "metadatas"]
)

print("ID:", result["ids"][0])
print("Embedding dimensions:", len(result["embeddings"][0]))
print("Document:", result["documents"][0][:300])
print("Metadata:", result["metadatas"][0])

In [ ]:
##Retrieval 


#Testing whether the database can retrieve the relevant chunks for a user query

In [ ]:
#setting a sample query
query = "PSMA targeted radiogland therapy in prostate cancer "

print(query)

In [ ]:
context = "\n\n".join(results["documents"][0])

print(context) 

In [ ]:
question = "PSMA targeted radioligand therapy in prostate cancer"

In [ ]:
prompt = f"""
You are a biomedical research assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
#getting an response from the local ollama model 

response = ollama.chat(
    model = "qwen2.5:3b",
    messages= [
        {

            "role": "user",
            "content": prompt
            
        }
    ]
)

answer = response["message"]["content"]

print(answer)

In [ ]:
#calculating the time taken by the model to give the response

import time 

start = time.time()
response = ollama.chat(
    model = "qwen2.5:3b",
    messages=[
        {
            "role":"user",
            "content": prompt
        }
    ]
)

end = time.time()
print("Generation time:", end - start, "seconds")
print(response["message"]["content"])



In [ ]:
import time

# 1. Query embedding time 
start = time.time()

query_embedding = ollama.embed(
    model="nomic-embed-text",
    input=question
)["embeddings"][0]

embedding_time = time.time() - start


# 2. Vector retrieval time
start = time.time()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

retrieval_time = time.time() - start


# 3. Context assembly time
start = time.time()

context = "\n\n".join(results["documents"][0])

context_time = time.time() - start


# 4. Prompt construction time
start = time.time()

prompt = f"""
You are a biomedical research assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

prompt_time = time.time() - start


print("Query embedding:", embedding_time, "seconds")
print("Retrieval:", retrieval_time, "seconds")
print("Context assembly:", context_time, "seconds")
print("Prompt construction:", prompt_time, "seconds")

In [ ]:
print("Prompt tokens:", response.get("prompt_eval_count"))
print("Generated tokens:", response.get("eval_count"))

print("Prompt evaluation time:",
      response.get("prompt_eval_duration", 0) / 1e9,
      "seconds")

print("Generation evaluation time:",
      response.get("eval_duration", 0) / 1e9,
      "seconds")

print("Total duration:",
      response.get("total_duration", 0) / 1e9,
      "seconds")

In [ ]:
#getting an response from the external LLM gemini 3.6
import ollama 
from dotenv import load_dotenv
import os 
from google import genai


load_dotenv()
 
api_key = os.getenv("GEMINI_API_KEY")
print("api key loaded:", api_key is not None)


client = genai.Client(api_key = api_key)
print("Gemini client created")

In [ ]:
question = "PSMA targeted radioligand therapy in prostate cancer"

query_embedding = ollama.embed(
    model="nomic-embed-text",
    input=question
)["embeddings"][0]

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

context = "\n\n".join(results["documents"][0])

prompt = f"""
You are a biomedical research assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

print("Retrieved chunks:", len(results["documents"][0]))
print("Prompt ready.")

In [ ]:
#checking the time taking by the gemini 3.6 model to provide an reponse

import time 

start = time.time()

response = client.models.generate_content(
    model = "gemini-3.6-flash",
    contents = prompt
)

end = time.time()

print("Gemini generation time:" , end - start , "seconds")
print()
print(response.text)   

In [ ]:
## RAG EVALUATION 

In [ ]:
evaluation_question = "What is PSMA-targeted radioligand therapy used for in prostate cancer?"

print(evaluation_question)

In [ ]:
query_embedding = ollama.embed(
    model="nomic-embed-text",
    input=evaluation_question
)["embeddings"][0]

eval_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

eval_context = "\n\n".join(eval_results["documents"][0])

print(eval_context) 

In [ ]:
evaluation_results = []

for item in evaluation_questions:

    question =  item["question"]

    query_embedding = ollama.embed(
        model = "nomic-embed-text",
        input = question

    )["embeddings"][0]


    results =  collection.query(

        query_embeddings = [query_embedding], 
        n_results = 5
    
    )


    evaluation_results.append({
        "id" : item["id"],
        "question": question,
        "category": item["category"],
        "retrieved_ids": results["ids"][0],
        "retrieved_documents": results["documents"][0],
        "distances": results["distances"][0]
    })

    print("Evaluated questions:" , len(evaluation_results))


    





In [ ]:
first_result = evaluation_results[0]

print("Question:", first_result["question"])
print()
print("Retrieved IDs:")
for i, doc_id in enumerate(first_result["retrieved_ids"], start=1):
    print(i, doc_id)

print()
print("Distances:")
for i, distance in enumerate(first_result["distances"], start=1):
    print(i, distance)

In [ ]:
import json 
import ollama 
import chromadb 


with open("../evaluation/evaluation_questions.json" , "r" , encoding = "utf-8") as f:
    evaluation_questions = json.load(f)

client_chroma = chromadb.PersistentClient(
    path = "../data/chroma"
)

collection = client_chroma.get_collection(

    name =  "documents"
)


print(collection.count())

In [ ]:
for doc in source_documents:
    if doc["doc_id"] != "38821587":
        print("\n" + "=" * 80)
        print(doc["doc_id"])
        print(doc["title"])
        print("=" * 80)
        print(doc["text"])

In [ ]:
import ollama

ground_truth_candidates = []

for item in candidate_sources:

    question = item["question"]

    # Build evidence from the full parent documents
    evidence_parts = []

    for doc in item["source_documents"]:
        evidence_parts.append(
            f"Source document: {doc['doc_id']}\n"
            f"Title: {doc['title']}\n"
            f"Text: {doc['text']}"
        )

    evidence = "\n\n".join(evidence_parts)

    prompt = f"""
You are helping create a golden reference answer for a biomedical RAG evaluation dataset.

Question:
{question}

Source documents:
{evidence}

Task:
Write a concise, factual reference answer to the question using ONLY information
supported by the source documents above.

Do not add outside medical knowledge.
Do not mention the source documents.
Do not speculate.
Do not make claims that are not supported by the provided text.

Reference answer:
"""

    response = ollama.chat(
        model="qwen2.5:3b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    candidate = response["message"]["content"].strip()

    ground_truth_candidates.append({
        "id": item["id"],
        "question": question,
        "ground_truth_candidate": candidate,
        "source_doc_ids": item["source_doc_ids"]
    })

    print(f"\n{'=' * 80}")
    print(f"Q{item['id']}")
    print(f"{'=' * 80}")
    print(candidate)
    print("\nSources:", item["source_doc_ids"])

In [ ]:
for item in candidate_sources:
    print("\n" + "#" * 100)
    print(f"Q{item['id']}: {item['question']}")
    print("#" * 100)

    for doc in item["source_documents"]:
        print("\n" + "=" * 90)
        print(f"DOC ID: {doc['doc_id']}")
        print(f"TITLE: {doc['title']}")
        print("=" * 90)
        print(doc["text"])

In [ ]:
import json
from pathlib import Path

ground_truth_path = Path("../evaluation/ground_truth.json")


with open(ground_truth_path , "r", encoding = "utf-8") as f:

    ground_truth = json.load(f)


new_ground_truth = {
     "2": {
        "question": "What is the role of 177Lu-PSMA-617 in prostate cancer treatment?",
        "answer": "177Lu-PSMA-617 is a PSMA-targeted radioligand therapy used primarily to treat metastatic castration-resistant prostate cancer (mCRPC), particularly in patients with sufficient PSMA expression. Evidence from the TheraP and VISION trials supports its safety and efficacy in post-chemotherapy mCRPC. Its use is also being investigated earlier in the treatment sequence.",
        "source_doc_ids": ["40769164", "40737767", "39692806", "39136842"]
    },
    "3": {
        "question": "What is the role of 225Ac-PSMA in prostate cancer?",
        "answer": "225Ac-PSMA is a targeted alpha radioligand therapy being investigated for advanced prostate cancer, particularly metastatic castration-resistant prostate cancer (mCRPC). It has shown encouraging therapeutic responses, including PSA declines and lesion regression, including in some heavily pre-treated patients. Current evidence remains more limited than for 177Lu-PSMA, and toxicity and treatment optimization remain important areas of investigation.",
        "source_doc_ids": ["40872606", "39552586", "39770150"]
    },
    "4": {
        "question": "How do PSMA-targeted radioligands work against prostate cancer?",
        "answer": "PSMA-targeted radioligands work by targeting PSMA, which is overexpressed on prostate cancer cells and is present in most metastatic lesions in castration-resistant prostate cancer. The radioligand binds to PSMA-expressing tumor sites and delivers therapeutic radiation to those cells. This targeted approach is used to selectively treat PSMA-positive prostate cancer lesions.",
        "source_doc_ids": ["40868092", "40647544", "40466338", "39770150"]
    },
    "5": {
        "question": "Why is PSMA useful as a target for radioligand therapy?",
        "answer": "PSMA is useful as a radioligand therapy target because it is highly expressed on the surface of prostate cancer cells while having more limited expression in normal tissue. Its presence on tumor cells allows small-molecule inhibitors or antibodies linked to radioactive isotopes to selectively target PSMA-expressing cancer tissue. PSMA expression can also be assessed with PSMA-targeted imaging, supporting patient selection for targeted therapy.",
        "source_doc_ids": ["NCT03490838", "40647544", "40427220", "39551117"]
    },
    "6": {
        "question": "What prostate cancer patients have been studied with PSMA-targeted radioligand therapy?",
        "answer": "PSMA-targeted radioligand therapy has primarily been studied in patients with advanced prostate cancer, particularly metastatic castration-resistant prostate cancer (mCRPC). Studies of 177Lu-PSMA-617 have included PSMA-positive patients, including those treated after chemotherapy. Research is also investigating these therapies earlier in the treatment sequence and evaluating emerging agents such as 225Ac-PSMA in selected advanced prostate cancer patients.",
        "source_doc_ids": ["42529062", "40868092", "39842443", "39770150"]
    }

}

ground_truth.update(new_ground_truth)

with open(ground_truth_path , "w" , encoding = "utf-8") as f:
    json.dump(ground_truth ,f, indent=2 , ensure_ascii=False)

with open(ground_truth_path , "r" , encoding = "utf-8") as f :
    validated = json.load(f)


print(f"ground_truth.json is valid JSON.")
print(f"Total entries: {len(validated)}")
print("Added/updated: Q2, Q3, Q4, Q5, Q6")





In [ ]:
import json
from pathlib import Path

ground_truth_path = Path("../evaluation/ground_truth.json")

ground_truth = {
    "2": {
        "question": "What is the role of 177Lu-PSMA-617 in prostate cancer treatment?",
        "answer": "177Lu-PSMA-617 is a PSMA-targeted radioligand therapy used primarily to treat metastatic castration-resistant prostate cancer (mCRPC), particularly in patients with sufficient PSMA expression. Evidence from the TheraP and VISION trials supports its safety and efficacy in post-chemotherapy mCRPC. Its use is also being investigated earlier in the treatment sequence.",
        "source_doc_ids": ["40769164", "40737767", "39692806", "39136842"]
    },
    "3": {
        "question": "What is the role of 225Ac-PSMA in prostate cancer?",
        "answer": "225Ac-PSMA is a targeted alpha radioligand therapy being investigated for advanced prostate cancer, particularly metastatic castration-resistant prostate cancer (mCRPC). It has shown encouraging therapeutic responses, including PSA declines and lesion regression, including in some heavily pre-treated patients. Current evidence remains more limited than for 177Lu-PSMA, and toxicity and treatment optimization remain important areas of investigation.",
        "source_doc_ids": ["40872606", "39552586", "39770150"]
    },
    "4": {
        "question": "How do PSMA-targeted radioligands work against prostate cancer?",
        "answer": "PSMA-targeted radioligands work by targeting PSMA, which is overexpressed on prostate cancer cells and is present in most metastatic lesions in castration-resistant prostate cancer. The radioligand binds to PSMA-expressing tumor sites and delivers therapeutic radiation to those cells. This targeted approach is used to selectively treat PSMA-positive prostate cancer lesions.",
        "source_doc_ids": ["40868092", "40647544", "40466338", "39770150"]
    },
    "5": {
        "question": "Why is PSMA useful as a target for radioligand therapy?",
        "answer": "PSMA is useful as a radioligand therapy target because it is highly expressed on the surface of prostate cancer cells while having more limited expression in normal tissue. Its presence on tumor cells allows small-molecule inhibitors or antibodies linked to radioactive isotopes to selectively target PSMA-expressing cancer tissue. PSMA expression can also be assessed with PSMA-targeted imaging, supporting patient selection for targeted therapy.",
        "source_doc_ids": ["NCT03490838", "40647544", "40427220", "39551117"]
    },
    "6": {
        "question": "What prostate cancer patients have been studied with PSMA-targeted radioligand therapy?",
        "answer": "PSMA-targeted radioligand therapy has primarily been studied in patients with advanced prostate cancer, particularly metastatic castration-resistant prostate cancer (mCRPC). Studies of 177Lu-PSMA-617 have included PSMA-positive patients, including those treated after chemotherapy. Research is also investigating these therapies earlier in the treatment sequence and evaluating emerging agents such as 225Ac-PSMA in selected advanced prostate cancer patients.",
        "source_doc_ids": ["42529062", "40868092", "39842443", "39770150"]
    }
}

with open(ground_truth_path, "w", encoding="utf-8") as f:
    json.dump(ground_truth, f, indent=2, ensure_ascii=False)

# Verify
with open(ground_truth_path, "r", encoding="utf-8") as f:
    test = json.load(f)

print("ground_truth.json created successfully.")
print(f"Entries: {len(test)}")
print("Questions:", list(test.keys()))

In [ ]:
import json 
from pathlib import Path

ground_truth_path = Path("../evaluation/ground_truth.json")

with open(ground_truth_path , "r" , encoding = "utf-8") as f:

    ground_truth.update({ 
        
        "7": {
        "question": "What evidence exists for PSMA-targeted radioligand therapy in metastatic castration-resistant prostate cancer?",
        "answer": (
            "PSMA-targeted radioligand therapy, particularly [177Lu]Lu-PSMA-617, "
            "has demonstrated clinical benefit in metastatic castration-resistant "
            "prostate cancer. Evidence from randomized trials and prospective "
            "studies, including TheraP and VISION, supports improvements in "
            "radiographic progression-free survival and PSA response, with an "
            "overall-survival benefit also reported."
        ),
        "source_doc_ids": [
            "40647544",
            "39842443",
            "40769164"
        ]
    },

    "8": {
        "question": "What are the reported safety considerations of PSMA-targeted radioligand therapy?",
        "answer": (
            "Reported safety considerations of PSMA-targeted radioligand therapy "
            "include salivary-gland toxicity such as xerostomia, hematologic "
            "toxicity, and potential renal toxicity. The severity and occurrence "
            "of these effects can vary with treatment and patient characteristics, "
            "making toxicity monitoring and management important during therapy."
        ),
        "source_doc_ids": [
            "39054909",
            "40647544",
            "NCT03490838"
        ]
    },

    "9": {
        "question": "What are the potential benefits of PSMA-targeted radiopharmaceuticals?",
        "answer": (
            "Potential benefits of PSMA-targeted radiopharmaceuticals include "
            "selective targeting of PSMA-expressing tumor tissue, improved imaging "
            "and disease localization, and the ability to combine diagnosis and "
            "therapy in a theranostic approach. Clinical studies of PSMA-targeted "
            "radioligand therapy have also reported improvements in PSA response "
            "and progression-related outcomes in appropriate advanced prostate "
            "cancer populations."
        ),
        "source_doc_ids": [
            "40647544",
            "39770150",
            "40448740"
        ]
    },

    "10": {
        "question": "What is the role of PSMA PET imaging in prostate cancer?",
        "answer": (
            "PSMA PET imaging is used to detect and localize PSMA-expressing "
            "prostate cancer and to support disease staging and management. In "
            "the radioligand-therapy setting, PSMA PET can help identify patients "
            "with sufficient PSMA expression for targeted treatment and can also "
            "contribute to assessment of treatment response."
        ),
        "source_doc_ids": [
            "40065665",
            "40418317",
            "40448740",
            "42279380"
        ]
    },

    "11": {
        "question": "Which PSMA-targeted radiopharmaceuticals have been used for prostate cancer imaging?",
        "answer": (
            "PSMA-targeted radiopharmaceuticals using different radionuclides have "
            "been used for prostate cancer imaging, including gallium-68-labeled "
            "PSMA agents for PET imaging. The corpus also describes other "
            "radiolabeled PSMA constructs under investigation for targeted imaging "
            "and theranostic applications."
        ),
        "source_doc_ids": [
            "39770150",
            "40868092",
            "40448740",
            "38733571"
        ]
    }
})

# Save updated ground truth
with open(ground_truth_path, "w", encoding="utf-8") as f:
    json.dump(ground_truth, f, indent=2, ensure_ascii=False)

# Verify
with open(ground_truth_path, "r", encoding="utf-8") as f:
    test = json.load(f)

print("ground_truth.json updated successfully.")
print(f"Entries: {len(test)}")
print("Questions:", sorted(test.keys(), key=int))


In [ ]:
# Q7–Q11: retrieve source documents for ground-truth creation

questions_7_11 = {
    "7": "What evidence exists for PSMA-targeted radioligand therapy in metastatic castration-resistant prostate cancer?",
    "8": "What are the reported safety considerations of PSMA-targeted radioligand therapy?",
    "9": "What are the potential benefits of PSMA-targeted radiopharmaceuticals?",
    "10": "What is the role of PSMA PET imaging in prostate cancer?",
    "11": "Which PSMA-targeted radiopharmaceuticals have been used for prostate cancer imaging?"
}

for q_id, question in questions_7_11.items():

    # Embed the question
    response = ollama.embed(
        model="nomic-embed-text",
        input=question
    )

    query_embedding = response["embeddings"][0]

    # Retrieve top 5 chunks
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    # Get chunk IDs
    chunk_ids = results["ids"][0]

    # Convert chunk IDs → parent document IDs
    parent_doc_ids = []
    for chunk_id in chunk_ids:
        parent_doc_id = chunk_id.rsplit("_", 1)[0]
        if parent_doc_id not in parent_doc_ids:
            parent_doc_ids.append(parent_doc_id)

    print("\n" + "=" * 80)
    print(f"Q{q_id}: {question}")
    print("=" * 80)

    print("\nRetrieved chunks:")
    for i, (chunk_id, distance) in enumerate(
        zip(chunk_ids, results["distances"][0]), 1
    ):
        print(f"{i}. {chunk_id} | distance={distance:.4f}")

    print("\nParent document IDs:")
    print(parent_doc_ids)

    print("\nDocument titles:")

    for doc_id in parent_doc_ids:
        matching_docs = [
            doc for doc in cleaned_documents
            if doc["doc_id"] == doc_id
        ]

        if matching_docs:
            print(f"- {doc_id}: {matching_docs[0]['title']}")
        else:
            print(f"- {doc_id}: NOT FOUND")

In [ ]:
# Embedding Questions #12-16

questions_12_16 = evaluation_questions[11:16]

for q in questions_12_16:
    print("\n" + "="*80)
    print(f"Q{q['id']}: {q['question']}")

    query_embedding = ollama.embed(
        model="nomic-embed-text",
        input=q["question"]
    )["embeddings"][0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    for i, (doc_id, distance) in enumerate(
        zip(results["ids"][0], results["distances"][0]), 1
    ):
        parent_id = doc_id.rsplit("_", 1)[0]
        print(f"{i}. {parent_id} | distance={distance:.4f}")

In [ ]:
questions = evaluation_questions[16:20]

for q in questions:
    response = ollama.embed(
        model="nomic-embed-text",
        input=q["question"]
    )

    query_embedding = response["embeddings"][0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    print("\n" + "=" * 80)
    print("QUESTION:", q["question"])

    for i, (doc, distance, metadata) in enumerate(zip(
        results["documents"][0],
        results["distances"][0],
        results["metadatas"][0]
    )):
        print(f"\n--- Result {i+1} | distance={distance:.4f} ---")
        print("ID:", metadata.get("id"))
        print(doc[:1000])

In [ ]:
import json
from pathlib import Path

ground_truth_path = Path("../evaluation/ground_truth.json")

with open(ground_truth_path, "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

ground_truth.update({
    "17": {
        "question": "What is the relationship between PSMA-targeted imaging and PSMA-targeted therapy?",
        "answer": (
            "PSMA-targeted imaging and therapy are linked through a theranostic "
            "approach: PSMA PET imaging can identify and characterize "
            "PSMA-expressing lesions, support patient selection for "
            "PSMA-targeted radioligand therapy, and contribute to response "
            "assessment. PSMA imaging can also provide information about "
            "changes in PSMA expression and heterogeneity during treatment "
            "and disease progression, which may affect therapeutic response "
            "and resistance."
        ),
        "source_doc_ids": [
            "40868092",
            "39842443"
        ]
    },

    "18": {
        "question": "What factors may affect the future clinical use of PSMA-targeted radioligand therapy?",
        "answer": (
            "Future clinical use of PSMA-targeted radioligand therapy may "
            "be affected by patient selection based on PSMA PET, treatment "
            "sequencing, management of salivary, hematologic, and renal "
            "toxicities, dosing and dosimetry, and integration into "
            "multidisciplinary care. Further randomized trials, individualized "
            "dosimetry, predictive biomarkers, toxicity-mitigation strategies, "
            "and evaluation of use in earlier disease stages are also important."
        ),
        "source_doc_ids": [
            "42529062",
            "40872606"
        ]
    },

    "19": {
        "question": "What evidence is available about PSMA-targeted radioligand therapy in cancers other than prostate cancer?",
        "answer": (
            "Evidence for PSMA-targeted radioligand therapy outside prostate "
            "cancer is limited and consists largely of early or case-based "
            "experience. A systematic review of the first 40 reported "
            "non-prostatic cases described use of 177Lu-PSMA in cancers "
            "including salivary-gland cancers, brain cancers, and osteosarcoma, "
            "with PSMA expression providing a potential treatment target. "
            "PSMA expression in some non-prostatic cancers may be associated "
            "with tumor neovasculature, providing another rationale for "
            "investigation."
        ),
        "source_doc_ids": [
            "38821587",
            "39551117"
        ]
    },

    "20": {
        "question": "What is the role of chemotherapy in treating bacterial infections?",
        "answer": (
            "This question is outside the scope of the prostate-cancer "
            "PSMA corpus, so no supported answer can be established from "
            "the retrieved evidence."
        ),
        "source_doc_ids": []
    }
})

with open(ground_truth_path, "w", encoding="utf-8") as f:
    json.dump(ground_truth, f, indent=2, ensure_ascii=False)

print("ground_truth.json updated successfully.")
print("Entries:", len(ground_truth))
print("Questions:", sorted(ground_truth.keys(), key=int))

In [ ]:
import rank_bm25
print("ye")

In [ ]:
# Load the existing chunks that were created during preprocessing.
# BM25 will use these same chunks that our dense retriever searches.
import json 

with open("../data/processed/chunked_documents.json" , "r" , encoding = "utf-8") as f:
    chunked_documents = json.load(f)


print("Number of documents:" , len(chunked_documents))


print("chunk1 : " , chunked_documents[0])

In [ ]:
tokenized_chunks = [
    chunk["text"].lower().split()
    for chunk in chunked_documents
]

print("Number of tokenzed chunks:" , len(tokenized_chunks))

print("First chunk tokens:" , tokenized_chunks[0][:20])



In [ ]:
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(tokenized_chunks)

print("Bm25 index created")

In [ ]:
# Use the same question that we used for dense retrieval.
# This lets us compare BM25 and Chroma fairly.
bm25_question = "PSMA targeted radioligand therapy in prostate cancer"

# Tokenize the query in the same way we tokenized the document chunks.
query_tokens = bm25_question.lower().split()

# Calculate a BM25 relevance score for every chunk.
bm25_scores = bm25.get_scores(query_tokens)

print("Number of BM25 scores:", len(bm25_scores))

In [ ]:
top_k = 5 

top_indices = sorted(
    range(len(bm25_scores)),
    key = lambda i: bm25_scores[i] , 
    reverse = True 
)[:top_k]

print("top bm25 chunk indices:" , top_indices)

In [ ]:
for rank, idx in enumerate(top_indices, start=1):
    chunk = chunked_documents[idx]

    print(f"\n--- BM25 Rank {rank} ---")
    print("BM25 Score:", round(bm25_scores[idx], 4))
    print("Chunk ID:", chunk["chunk_id"])
    print("Document ID:", chunk["doc_id"])
    print("Source:", chunk["source"])
    print("Text:", chunk["text"][:500])

In [ ]:
# Retrieve the Top-10 chunks using BM25.
# We retrieve more candidates than our final Top-K so that RRF
# has a larger pool of results to combine.

retrieval_k = 10

bm25_top_indices = sorted(
    range(len(bm25_scores)),
    key=lambda i: bm25_scores[i],
    reverse=True
)[:retrieval_k]

print("BM25 Top-10 indices:")
print(bm25_top_indices)

In [ ]:
# Retrieve the Top-10 chunks using dense vector search.
# This uses the same query embedding we created earlier.

dense_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=retrieval_k
)

print("Dense Top-10 retrieved.")

In [ ]:
bm25_ranked_ids = [
    chunked_documents[idx]["chunk_id"]
    for idx in bm25_top_indices
]

dense_ranked_ids = dense_results["ids"][0]


print("bm25 ranked ids:")
print(bm25_ranked_ids)
("\nDense ranked ids:")

print(dense_ranked_ids)

In [ ]:
# Reciprocal Rank Fusion (RRF)
# RRF combines ranked results from different retrieval systems.
# It uses the rank position rather than the original retrieval score.

rrf_k = 60

rrf_scores = {}

# Add contributions from the BM25 ranking.
for rank, chunk_id in enumerate(bm25_ranked_ids, start=1):
    rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + (
        1 / (rrf_k + rank)
    )

# Add contributions from the dense ranking.
for rank, chunk_id in enumerate(dense_ranked_ids, start=1):
    rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + (
        1 / (rrf_k + rank)
    )

print("Number of unique candidates:", len(rrf_scores))

In [ ]:
# Sort all candidates by their combined RRF score.
# Higher RRF score means the chunk received stronger ranking evidence.

rrf_ranked = sorted(
    rrf_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

print("Top RRF results:")

for rank, (chunk_id, score) in enumerate(rrf_ranked[:10], start=1):
    print(
        f"Rank {rank}: {chunk_id} | "
        f"RRF Score: {score:.6f}"
    )

In [ ]:
# Load the evaluation ground truth.
# The ground truth tells us which source documents are relevant
# for each evaluation question.

with open("../evaluation/ground_truth.json", "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

print("Number of ground-truth questions:", len(ground_truth))
print("Question 1:")
print(ground_truth["1"])

In [ ]:
# Create a mapping from each chunk_id to its parent document ID.
# Retrieval happens at chunk level, but our ground truth is defined
# using parent/source document IDs.

chunk_to_doc = {
    chunk["chunk_id"]: chunk["doc_id"]
    for chunk in chunked_documents
}

print("Number of chunk mappings:", len(chunk_to_doc))

# Example mapping
for chunk_id in list(chunk_to_doc.keys())[:5]:
    print(chunk_id, "→", chunk_to_doc[chunk_id])

In [ ]:
# Map each chunk ID to its parent document ID.
# The chunk IDs are used by retrieval, while the parent
# document IDs are used by our ground-truth evaluation.

chunk_to_doc = {
    chunk["chunk_id"]: chunk["doc_id"]
    for chunk in chunked_documents
}

print("Number of chunk mappings:", len(chunk_to_doc))

In [ ]:
#creating mappings from chunked_documents.json 

chunk_to_doc = {
    chunk["chunk_id"]: chunk["doc_id"]
    for chunk in chunked_documents
}

print("number of mappings:" , len(chunk_to_doc))

In [ ]:
import json
from pathlib import Path

ground_truth_path = Path("../evaluation/ground_truth.json")


with open(ground_truth_path , "r", encoding = "utf-8") as f:

    ground_truth = json.load(f)

In [ ]:
q1 =  ground_truth["1"]

print("Question:")
print(q1["question"])

print("\n relevant document ids :")

print(q1["source_doc_ids"])


In [ ]:
retrieved_chunk_ids = [
    "40647544_2",
    "39770150_2",
    "39770150_1",
    "40868092_1",
    "38821587_2"
]

retrieved_doc_ids = [
    chunk_to_doc[chunk_id]
    for chunk_id in retrieved_chunk_ids
]

retrieved_docs = set(retrieved_doc_ids)
relevant_docs = set(q1["source_doc_ids"])

print("Retrieved document IDs:", retrieved_docs)
print("Relevant document IDs:", relevant_docs)

intersection = retrieved_docs & relevant_docs

print("Relevant documents retrieved:", intersection)
print("Number retrieved:", len(intersection))
print("Number relevant:", len(relevant_docs))


recall_at_5 = len(intersection) / len(relevant_docs)

print("Recall@5" , recall_at_5) 

In [ ]:
def recall_at_k(retrieved_chunk_ids, relevant_doc_ids, k):
    """
    Calculate document-level Recall@K.

    retrieved_chunk_ids: ranked list of retrieved chunk IDs
    relevant_doc_ids: ground-truth parent document IDs
    k: number of top retrieved chunks to consider
    """

    # Take only the top K retrieved chunks
    top_k_chunks = retrieved_chunk_ids[:k]

    # Convert chunk IDs -> parent document IDs
    retrieved_doc_ids = [
        chunk_to_doc[chunk_id]
        for chunk_id in top_k_chunks
    ]

    # Deduplicate parent documents
    retrieved_docs = set(retrieved_doc_ids)
    relevant_docs = set(relevant_doc_ids)

    # Find relevant documents that were actually retrieved
    intersection = retrieved_docs & relevant_docs

    # Recall@K
    recall = len(intersection) / len(relevant_docs)

    return recall

In [ ]:
q1_retrieved = [
    "40647544_2",
    "39770150_2",
    "39770150_1",
    "40868092_1",
    "38821587_2"
]

q1_relevant = ground_truth["1"]["source_doc_ids"]

print("Recall@1:", recall_at_k(q1_retrieved, q1_relevant, 1))
print("Recall@3:", recall_at_k(q1_retrieved, q1_relevant, 3))
print("Recall@5:", recall_at_k(q1_retrieved, q1_relevant, 5))

In [ ]:
with open("../evaluation/ground_truth.json" , "r" , encoding = "utf-8") as f:
    ground_truth = json.load(f)

    

In [ ]:
def retrieve_chunks(query, k=10):
    response = ollama.embeddings(
        model="nomic-embed-text",
        prompt=query
    )

    query_embedding = response["embedding"]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["ids"][0]

In [ ]:
recall_results = {}

for question_id, data in ground_truth.items():

    question = data["question"]
    relevant_doc_ids = data["source_doc_ids"]

    # Q20 has no supported ground-truth documents
    if not relevant_doc_ids:
        recall_results[question_id] = {
            "Recall@1": None,
            "Recall@3": None,
            "Recall@5": None,
            "Recall@10": None
        }
        continue

    # Retrieve top 10 chunks
    retrieved_chunks = retrieve_chunks(question, k=10)

    # Evaluate
    recall_results[question_id] = {
        "Recall@1": recall_at_k(
            retrieved_chunks,
            relevant_doc_ids,
            1
        ),
        "Recall@3": recall_at_k(
            retrieved_chunks,
            relevant_doc_ids,
            3
        ),
        "Recall@5": recall_at_k(
            retrieved_chunks,
            relevant_doc_ids,
            5
        ),
        "Recall@10": recall_at_k(
            retrieved_chunks,
            relevant_doc_ids,
            10
        )
    }

# Display results
for question_id, results in recall_results.items():
    print(f"Q{question_id}: {results}")

In [ ]:
for question_id in ["17", "18", "19"]:

    question = ground_truth[question_id]["question"]
    relevant_docs = set(
        ground_truth[question_id]["source_doc_ids"]
    )

    retrieved_chunks = retrieve_chunks(question, k=10)

    retrieved_docs = {
        chunk_to_doc[chunk_id]
        for chunk_id in retrieved_chunks
    }

    print("=" * 70)
    print(f"Q{question_id}")
    print("\nQuestion:")
    print(question)

    print("\nRelevant documents:")
    print(relevant_docs)

    print("\nRetrieved chunks:")
    print(retrieved_chunks)

    print("\nRetrieved documents:")
    print(retrieved_docs)

    print("\nRelevant documents retrieved:")
    print(retrieved_docs & relevant_docs)

In [ ]:
valid_results = [
    result
    for result in recall_results.values()
    if result["Recall@5"] is not None
]

for k in [1, 3, 5, 10]:
    key = f"Recall@{k}"
    average_recall = sum(
        result[key] for result in valid_results
    ) / len(valid_results)

    print(f"Average {key}: {average_recall:.4f}")

In [ ]:
def precision_at_k(retrieved_chunk_ids, relevant_doc_ids, k):
    top_k_chunks = retrieved_chunk_ids[:k]

    retrieved_doc_ids = [
        chunk_to_doc[chunk_id]
        for chunk_id in top_k_chunks
    ]

    relevant_docs = set(relevant_doc_ids)

    relevant_retrieved = sum(
        1
        for doc_id in retrieved_doc_ids
        if doc_id in relevant_docs
    )

    precision = relevant_retrieved / k

    return precision

In [ ]:
precision_results = {}

for question_id, data in ground_truth.items():

    question = data["question"]
    relevant_doc_ids = data["source_doc_ids"]

    if not relevant_doc_ids:
        precision_results[question_id] = {
            "Precision@1": None,
            "Precision@3": None,
            "Precision@5": None,
            "Precision@10": None
        }
        continue

    retrieved_chunks = retrieve_chunks(question, k=10)

    precision_results[question_id] = {
        "Precision@1": precision_at_k(
            retrieved_chunks,
            relevant_doc_ids,
            1
        ),
        "Precision@3": precision_at_k(
            retrieved_chunks,
            relevant_doc_ids,
            3
        ),
        "Precision@5": precision_at_k(
            retrieved_chunks,
            relevant_doc_ids,
            5
        ),
        "Precision@10": precision_at_k(
            retrieved_chunks,
            relevant_doc_ids,
            10
        )
    }

In [ ]:
for question_id , results in precision_results.items():
    print(f"Q{question_id}: {results}")
     

In [ ]:
valid_precision_results = [
    result
    for result in precision_results.values()
    if result["Precision@5"] is not None
]

for k in [1, 3, 5, 10]:

    key = f"Precision@{k}"

    average_precision = sum(
        result[key]
        for result in valid_precision_results
    ) / len(valid_precision_results)

    print(f"Average {key}: {average_precision:.4f}")

In [ ]:
f1_results = {}

for question_id in ground_truth:

    if recall_results[question_id]["Recall@5"] is None:
        f1_results[question_id] = {
            "F1@1": None,
            "F1@3": None,
            "F1@5": None,
            "F1@10": None
        }
        continue

    f1_results[question_id] = {}

    for k in [1, 3, 5, 10]:

        recall = recall_results[question_id][f"Recall@{k}"]
        precision = precision_results[question_id][f"Precision@{k}"]

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0
            else 0
        )

        f1_results[question_id][f"F1@{k}"] = f1

In [ ]:
for k in [1, 3, 5, 10]:

    key = f"F1@{k}"

    values = [
        result[key]
        for result in f1_results.values()
        if result[key] is not None
    ]

    average_f1 = sum(values) / len(values)

    print(f"Average {key}: {average_f1:.4f}")

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [
    chunk["text"].lower().split()
    for chunk in chunked_documents
]

bm25 = BM25Okapi(tokenized_chunks)

In [ ]:
#chunk indices

chunk_index_to_id = {
i: chunk["chunk_id"]
for i , chunk in enumerate(chunked_documents)

}

In [ ]:
def retrieve_bm25(query , k=10):
    scores = bm25.get_scores(query.lower().split())

    top_indices = scores.argsort()[::-1][:k]

    return [ 
        chunk_index_to_id[index]
        for index in top_indices
    ]

In [ ]:
query = "PSMA targeted radioligand therapy in prostate cancer"

bm25_results = retrieve_bm25(query, k=5)

print(bm25_results)

In [ ]:
def reciprocal_rank_fusion(
    dense_results,
    bm25_results,
    rrf_k=60
):
    scores = {}

    for rank, chunk_id in enumerate(dense_results, start=1):
        scores[chunk_id] = scores.get(chunk_id, 0) + (
            1 / (rrf_k + rank)
        )

    for rank, chunk_id in enumerate(bm25_results, start=1):
        scores[chunk_id] = scores.get(chunk_id, 0) + (
            1 / (rrf_k + rank)
        )

    ranked_results = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [chunk_id for chunk_id, score in ranked_results]

In [ ]:
question = ground_truth["1"]["question"]

dense_results = retrieve_chunks(
    question,
    k=10
)

bm25_results = retrieve_bm25(
    question,
    k=10
)

hybrid_results = reciprocal_rank_fusion(
    dense_results,
    bm25_results
)

print("Dense:")
print(dense_results)

print("\nBM25:")
print(bm25_results)

print("\nHybrid:")
print(hybrid_results[:10])

In [ ]:
def reciprocal_rank_fusion_with_scores(
    dense_results,
    bm25_results,
    rrf_k=60
):
    scores = {}

    for rank, chunk_id in enumerate(dense_results, start=1):
        scores[chunk_id] = scores.get(chunk_id, 0) + (
            1 / (rrf_k + rank)
        )

    for rank, chunk_id in enumerate(bm25_results, start=1):
        scores[chunk_id] = scores.get(chunk_id, 0) + (
            1 / (rrf_k + rank)
        )

    ranked_results = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked_results

In [ ]:
hybrid_scored = reciprocal_rank_fusion_with_scores(
    dense_results,
    bm25_results
)

for rank, (chunk_id, score) in enumerate(
    hybrid_scored[:10],
    start=1
):
    print(rank, chunk_id, score)

In [ ]:
def retrieve_hybrid(query, k=10):
    dense_results = retrieve_chunks(query, k=k)
    bm25_results = retrieve_bm25(query, k=k)

    hybrid_results = reciprocal_rank_fusion(
        dense_results,
        bm25_results
    )

    return hybrid_results[:k]

In [ ]:
hybrid_recall_results = {}

for question_id, data in ground_truth.items():

    question = data["question"]
    relevant_doc_ids = data["source_doc_ids"]

    if not relevant_doc_ids:
        hybrid_recall_results[question_id] = {
            "Recall@1": None,
            "Recall@3": None,
            "Recall@5": None,
            "Recall@10": None
        }
        continue

    retrieved_chunks = retrieve_hybrid(
        question,
        k=10
    )

    hybrid_recall_results[question_id] = {
        "Recall@1": recall_at_k(
            retrieved_chunks,
            relevant_doc_ids,
            1
        ),
        "Recall@3": recall_at_k(
            retrieved_chunks,
            relevant_doc_ids,
            3
        ),
        "Recall@5": recall_at_k(
            retrieved_chunks,
            relevant_doc_ids,
            5
        ),
        "Recall@10": recall_at_k(
            retrieved_chunks,
            relevant_doc_ids,
            10
        )
    }

In [ ]:
for k in [1, 3, 5, 10]:

    key = f"Recall@{k}"

    values = [
        result[key]
        for result in hybrid_recall_results.values()
        if result[key] is not None
    ]

    average_recall = sum(values) / len(values)

    print(f"Hybrid Average {key}: {average_recall:.4f}")

In [ ]:
for question_id in ground_truth:

    dense = recall_results[question_id]["Recall@5"]
    hybrid = hybrid_recall_results[question_id]["Recall@5"]

    if dense is None:
        continue

    difference = hybrid - dense

    print(
        f"Q{question_id}: "
        f"Dense={dense:.3f}, "
        f"Hybrid={hybrid:.3f}, "
        f"Change={difference:+.3f}"
    )

In [ ]:
question_id = "2"

question = ground_truth[question_id]["question"]
relevant_doc_ids = set(
    ground_truth[question_id]["source_doc_ids"]
)

dense_results = retrieve_chunks(question, k=10)
bm25_results = retrieve_bm25(question, k=10)
hybrid_results = retrieve_hybrid(question, k=10)

print("QUESTION:")
print(question)

print("\nRELEVANT DOCS:")
print(relevant_doc_ids)

print("\nDENSE:")
for rank, chunk_id in enumerate(dense_results, start=1):
    print(
        rank,
        chunk_id,
        "DOC:", chunk_to_doc[chunk_id],
        "RELEVANT:", chunk_to_doc[chunk_id] in relevant_doc_ids
    )

print("\nBM25:")
for rank, chunk_id in enumerate(bm25_results, start=1):
    print(
        rank,
        chunk_id,
        "DOC:", chunk_to_doc[chunk_id],
        "RELEVANT:", chunk_to_doc[chunk_id] in relevant_doc_ids
    )

print("\nHYBRID:")
for rank, chunk_id in enumerate(hybrid_results, start=1):
    print(
        rank,
        chunk_id,
        "DOC:", chunk_to_doc[chunk_id],
        "RELEVANT:", chunk_to_doc[chunk_id] in relevant_doc_ids
    )

In [ ]:
def retrieve_candidate_pool(query, k=10):
    dense_results = retrieve_chunks(query, k=k)
    bm25_results = retrieve_bm25(query, k=k)

    candidates = list(dict.fromkeys(
        dense_results + bm25_results
    ))

    return candidates

In [ ]:
candidate_recall_results = {}

for question_id, data in ground_truth.items():

    relevant_doc_ids = data["source_doc_ids"]

    if not relevant_doc_ids:
        candidate_recall_results[question_id] = None
        continue

    candidates = retrieve_candidate_pool(
        data["question"],
        k=10
    )

    candidate_recall_results[question_id] = recall_at_k(
        candidates,
        relevant_doc_ids,
        len(candidates)
    )

In [ ]:
valid_results = [
    value
    for value in candidate_recall_results.values()
    if value is not None
]

sum(valid_results) / len(valid_results)

In [ ]:
question_id = "2"

query = ground_truth[question_id]["question"]

candidates = retrieve_candidate_pool(
    query,
    k=10
)

print("QUERY:")
print(query)

print("\nCANDIDATES:")

for rank, chunk_id in enumerate(candidates, start=1):

    chunk = next(
        c for c in chunked_documents
        if c["chunk_id"] == chunk_id
    )

    print("\n" + "=" * 80)
    print(f"CANDIDATE {rank}")
    print(f"Chunk ID: {chunk_id}")
    print(f"Document ID: {chunk['doc_id']}")
    print("-" * 80)
    print(chunk["text"][:1000])

In [ ]:
import re

def simple_rerank(query, candidate_ids):
    query_terms = set(
        re.findall(r"\b[a-zA-Z0-9]+\b", query.lower())
    )

    scored = []

    for chunk_id in candidate_ids:

        chunk = next(
            c for c in chunked_documents
            if c["chunk_id"] == chunk_id
        )

        text_terms = set(
            re.findall(r"\b[a-zA-Z0-9]+\b", chunk["text"].lower())
        )

        overlap = query_terms & text_terms

        score = len(overlap) / len(query_terms)

        scored.append((chunk_id, score))

    scored.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return scored

In [ ]:
query = ground_truth["2"]["question"]

candidates = retrieve_candidate_pool(
    query,
    k=10
)

reranked = simple_rerank(
    query,
    candidates
)

for rank, (chunk_id, score) in enumerate(
    reranked,
    start=1
):
    print(
        rank,
        chunk_id,
        f"score={score:.3f}",
        "DOC:",
        chunk_to_doc[chunk_id]
    )

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

model.eval()

print("Reranker loaded successfully")

In [ ]:
def rerank(query, candidate_chunks, top_k=5):
    pairs = [
        (query, chunk["text"])
        for chunk in candidate_chunks
    ]

    inputs = tokenizer(
        pairs,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    scores = outputs.logits.squeeze(-1)

    ranked = sorted(
        zip(candidate_chunks, scores.tolist()),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_k]

In [ ]:
question_id = "2"

query = ground_truth[question_id]["question"]

candidate_ids = retrieve_candidate_pool(
    query,
    k=10
)

candidate_chunks = [
    next(
        c for c in chunked_documents
        if c["chunk_id"] == chunk_id
    )
    for chunk_id in candidate_ids
]

In [ ]:
reranked = rerank(
    query,
    candidate_chunks,
    top_k=5
)

In [ ]:
for rank, (chunk, score) in enumerate(reranked, start=1):
    print("=" * 100)
    print(f"RANK {rank} | SCORE {score:.4f}")
    print(f"CHUNK: {chunk['chunk_id']}")
    print(f"DOC:   {chunk['doc_id']}")
    print()
    print(chunk["text"][:1200])